# Itemset — native Julia on Colab

Select **Runtime → Change runtime type → Julia**. All code cells run directly in the Julia kernel. Use Julia 1.12+ within 1.x; if needed select runtime version **2026.07** (Julia 1.12.6) or a newer compatible runtime. See [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html).

For this local repository, export a committed ZIP with `git archive --format=zip --output=../Itemset-colab.zip HEAD`, upload it through Colab's **Files** sidebar to `/content/Itemset-colab.zip`, and run all cells. Alternatively set a future public GitHub URL and exact commit below.

The notebook verifies the environment and tests, then runs the Semeion image pilot. Keep the ZIP revision with the output.

In [ ]:
REPO_URL = ""  # Optional future public GitHub HTTPS URL.
REVISION = ""  # Exact commit required for cloning; record the ZIP's commit too.
ROOT = get(ENV, "ITEMSET_ROOT", isdir("/content") ? "/content/Itemset" : pwd())
ARCHIVE = "/content/Itemset-colab.zip"  # Upload through Colab's Files sidebar.

if !isempty(REPO_URL)
    isempty(REVISION) && error("Set an exact commit in REVISION for reproducibility")
    isdir(joinpath(ROOT, ".git")) || run(`git clone $REPO_URL $ROOT`)
    run(`git -C $ROOT checkout --detach $REVISION`)
elseif !isfile(joinpath(ROOT, "Project.toml"))
    isfile(ARCHIVE) || error("Upload Itemset-colab.zip through the Files sidebar first")
    mkpath(ROOT)
    run(`unzip -q -o $ARCHIVE -d $ROOT`)
end

isfile(joinpath(ROOT, "Project.toml")) || error("Repository root must contain Project.toml")
ROOT = abspath(ROOT)
cd(ROOT)
println("Repository: ", ROOT)
println("Revision: ", isdir(".git") ? readchomp(`git rev-parse HEAD`) :
        isempty(REVISION) ? "local archive; record its commit before final experiments" : REVISION)


In [ ]:
using Pkg
v"1.12" <= VERSION < v"2" ||
    error("Select a Colab Julia 1.12+ runtime (e.g. runtime version 2026.07)")
Pkg.activate(ROOT)
Pkg.instantiate()

using ItemsetMining
using Random
println("Julia: ", VERSION, " | threads: ", Threads.nthreads(), " | seed: ", DEFAULT_SEED)
println("Project: ", Base.active_project())


In [ ]:
# Execute setup and tests directly inside this Julia kernel.
# Separate modules keep test definitions out of the notebook workspace.
Base.include(Module(:ItemsetSetupCheck), joinpath(ROOT, "scripts", "check_setup.jl"))
module ItemsetSetupTests
    include(joinpath(Main.ROOT, "test", "runtests.jl"))
end


In [ ]:
transactions = read_transactions(joinpath(ROOT, "data", "toy", "baskets.txt"))
minsup_count = support_threshold(0.5, length(transactions))
(; transactions, minsup_count)


## Hung — Eclat, tests, scalability, output size

`mine` with `variant=:baseline` runs the paper-based Eclat described in `docs/eclat.md`. The cell below mines the toy baskets at minsup 0.5 and prints the SPMF output, which matches `data/toy/baskets.expected.txt`. The test cell above checks the baseline against every toy fixture and a brute-force count.

In [ ]:
patterns = mine(transactions; minsup=0.5, variant=:baseline)
write_itemsets(stdout, patterns)

## Thien — Benchmarks, market baskets, runtime, transaction width

The next cell downloads the four benchmarks, checks each file's SHA-256 against `data/benchmark/manifest.toml`, and prints the statistics computed from the files. Timing runs use `experiments/run.jl` as described in `experiments/README.md`; SPMF series need Java 25 or later.

The cell after it downloads Online Retail II, checks its SHA-256 against `data/raw/manifest.toml`, cleans it into one basket per invoice, and writes the baskets and the stock-code mapping to `data/raw/market_basket/`. `applications/market_basket/README.md` explains each cleaning step.
Rule tests, top-10 lift rules and interpretation (DAT-14/37), then runtime and width plots follow. Stochastic steps use `DEFAULT_SEED`.

In [ ]:
# Prepare and verify the benchmarks in data/benchmark/ (about 9 MB of downloads).
module PrepareBenchmarks
    include(joinpath(Main.ROOT, "experiments", "prepare_benchmarks.jl"))
end
PrepareBenchmarks.main(String[])


In [ ]:
# Clean Online Retail II into invoice baskets (about 95 MB download).
module PrepareBaskets
    include(joinpath(Main.ROOT, "applications", "market_basket", "prepare.jl"))
end
PrepareBaskets.main(String[])
items = split.(readlines(joinpath(ROOT, "data", "raw", "market_basket", "items.tsv"))[2:end], '\t')
first(sort(items; by=row -> -parse(Int, row[4])), 10)


## Phat: deterministic Semeion encoding

The [UCI Semeion dataset](https://archive.ics.uci.edu/dataset/178/semeion+handwritten+digit) contains binary 16×16 handwritten digits under CC BY 4.0. The preprocessing cell verifies the source, fits a private seed-42 patch codebook, and keeps one distinct-word transaction per image, including empty cases. `manifest.toml` records settings, revision, source and artifact hashes, width distribution and density. SVGs show observed patches assigned to their words. The whole dataset fits this descriptive codebook; this is not a generalization evaluation. Position and multiplicity are lost in the word sets. The following pilot cell mines the same encoding at three thresholds.

In [ ]:
semeion_path = joinpath(ROOT, "data", "raw", "semeion", "semeion.data")
encoding_output = joinpath(mktempdir(), "semeion-encoding")
module ImagePilotRun
    include(joinpath(Main.ROOT, "scripts", "image_pilot.jl"))
end
ImagePilotRun.encoding_main([semeion_path, encoding_output])
encoding_manifest = ImagePilotRun.SemeionPilot.TOML.parsefile(joinpath(encoding_output, "manifest.toml"))
display(encoding_manifest["statistics"])
display(HTML(read(joinpath(encoding_output, "digits.svg"), String)))
display(HTML(read(joinpath(encoding_output, "visual_words.svg"), String)))


In [ ]:
pilot_output = joinpath(mktempdir(), "semeion-pilot")
ImagePilotRun.main([semeion_path, pilot_output])
println(read(joinpath(pilot_output, "thresholds.tsv"), String))


### Independent SPMF Eclat check

A local Windows run compared complete itemset/support maps from SPMF Eclat with the Julia baseline on five toy databases. The boundary database was checked at count thresholds 1 and 2. Every comparison found zero missing, extra, duplicate, or support-mismatched itemsets. These are local reference results; they do not claim a Colab or benchmark run.

| Database | Transactions | Count threshold | Exact pairs | Match rate |
| --- | ---: | ---: | ---: | ---: |
| Baskets | 4 | 2 | 6/6 | 100% |
| Dense | 2 | 2 | 7/7 | 100% |
| Groceries | 5 | 2 | 5/5 | 100% |
| Boundary | 4 | 2 | 2/2 | 100% |
| Boundary | 4 | 1 | 4/4 | 100% |
| Nested | 4 | 2 | 3/3 | 100% |

The match rate counts itemsets with equal support in both outputs and divides by the number of distinct itemsets in their union. See `docs/spmf-toy-validation-2026-10-24.md` for versions, checksums, and the raw-evidence location.

## Clean-run checklist

Restart the Julia runtime and run all cells after modules are implemented. Record the Colab runtime version, Julia version, thread count, revision, dataset hashes, dependencies, parameters, stdout, and failures. Publish a public Colab link in README after verifying access and a complete clean run. A local run does not prove the hosted Colab runtime has been exercised.